# 趋势追踪行业配置策略 - 研报复现

**研报来源**: 华泰证券 - 基本面轮动系列之七：行业配置策略：趋势追踪视角 (2020-08-31)

本notebook复现研报中的核心内容：
1. 趋势追踪指标体系构建
2. 蒙特卡洛模拟分析
3. 大类资产配置策略
4. 行业轮动策略
5. CSCV过拟合检验

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

from source import (
    DataLoader, TrendIndicatorCalculator, BacktestEngine,
    MonteCarloSimulator, CSCVTest, StrategyConfig,
    TrendFollowingStrategyBuilder
)

print('模块导入成功!')

## 1. 数据获取

In [ ]:
# 使用tushare获取数据
loader = DataLoader()

# 获取主要指数数据
index_data = loader.get_multiple_indices(
    start_date='20100101',
    end_date='20200731'
)
print(f'\n成功获取 {len(index_data)} 个指数数据')

In [ ]:
# 获取行业指数数据
industry_data = loader.get_multiple_industries(
    start_date='20100101',
    end_date='20200731'
)
print(f'成功获取 {len(industry_data)} 个行业数据')

In [ ]:
# 计算资产统计信息
index_stats = loader.calculate_assets_statistics(index_data)
print('\n指数资产统计信息:')
print(index_stats.to_string())

In [ ]:
# 计算相关系数矩阵
corr_matrix = loader.calculate_correlation_matrix(index_data)
print('\n指数相关系数矩阵 (部分):')
print(corr_matrix.iloc[:5, :5].round(3))

## 2. 趋势追踪指标计算

In [ ]:
# 初始化指标计算器
calculator = TrendIndicatorCalculator()
print(f'实现趋势追踪指标数量: {len(calculator.indicator_funcs)}')

In [ ]:
# 以沪深300为例计算指标
hs300 = index_data['沪深300'].copy()
hs300['returns'] = hs300['close'].pct_change()

# 生成单一指标信号示例
ema_20 = calculator.generate_signal(hs300, 'EMA', {'n': 20})
macd_signal = calculator.generate_signal(hs300, 'MACD', {'n': 12, 'n2': 26, 'n3': 9})
dpo_20 = calculator.generate_signal(hs300, 'DPO', {'n': 20})

print('EMA(20) 信号示例:')
print(ema_20.tail(10))

In [ ]:
# 生成全部指标信号
all_signals = calculator.generate_all_signals(hs300)
print(f'共生成 {len(all_signals.columns)} 个指标信号')

## 3. 蒙特卡洛模拟分析

In [ ]:
# 单资产蒙特卡洛模拟
simulator = MonteCarloSimulator(random_state=42)

# 参数设置 (参考研报)
mu_range = [0.0001, 0.0005, 0.001]  # 日收益率
sigma_range = [0.005, 0.01, 0.02]   # 日波动率
rho_range = [0.01, 0.1, 0.2]       # 自相关系数

print('单资产蒙特卡洛模拟参数:')
print(f'收益率范围: {mu_range}')
print(f'波动率范围: {sigma_range}')
print(f'自相关系数范围: {rho_range}')

In [ ]:
# 生成单资产虚拟序列
n_days = 100
n_scenarios = 100

virtual_sequences = simulator.generate_single_asset_scenarios(
    n_days=n_days,
    n_scenarios=n_scenarios,
    mu_range=mu_range,
    sigma_range=sigma_range,
    rho_range=rho_range
)
print(f'生成了 {len(virtual_sequences)} 个虚拟序列场景')

In [ ]:
# 多资产几何布朗运动模拟
n_assets = 3
mus = np.array([0.0005, 0.0003, 0.0004])
sigmas = np.array([0.015, 0.01, 0.012])
corr = np.array([[1.0, 0.3, 0.2],
                  [0.3, 1.0, 0.25],
                  [0.2, 0.25, 1.0]])

multi_asset_paths = simulator.generate_multi_asset_gbm(
    n_days=n_days,
    n_assets=n_assets,
    mu=mus,
    sigma=sigmas,
    correlation_matrix=corr
)
print(f'多资产价格路径形状: {multi_asset_paths.shape}')

## 4. 策略回测

In [ ]:
# 准备价格数据
prices_df = pd.DataFrame({name: df.set_index('trade_date')['close'] 
                          for name, df in index_data.items()})
prices_df = prices_df.sort_index()
prices_df = prices_df.dropna()

print(f'价格数据形状: {prices_df.shape}')
print(f'时间范围: {prices_df.index[0]} 至 {prices_df.index[-1]}')

In [ ]:
# 初始化回测引擎
engine = BacktestEngine(
    rebalance_freq='monthly',
    commission_rate=0.001
)

# 评估单一指标 (以EMA20为例)
ema_20_signal = calculator.generate_signal(hs300, 'EMA', {'n': 20})
ema_20_signal_aligned = ema_20_signal.reindex(prices_df.index, method='ffill')

result_ema20 = engine.evaluate_indicator(
    prices_df, ema_20_signal_aligned,
    strategy_type='time_series'
)
print('EMA(20) 策略表现:')
print(f'  年化收益率: {result_ema20["annual_return"]:.2%}')
print(f'  年化波动率: {result_ema20["annual_volatility"]:.2%}')
print(f'  夏普比率: {result_ema20["sharpe_ratio"]:.4f}')
print(f'  最大回撤: {result_ema20["max_drawdown"]:.2%}')

In [ ]:
# 批量评估指标
evaluator = StrategyEvaluator()

# 选取部分指标进行评估 (完整评估较耗时)
sample_signals = all_signals[['EMA_20', 'EMA_40', 'EMA_60', 
                              'MACD_12_26_9', 'ROC_20', 'DPO_20']].copy()
sample_signals_aligned = sample_signals.reindex(prices_df.index, method='ffill')

results_df = evaluator.evaluate_multiple_indicators(
    prices_df, sample_signals_aligned,
    strategy_type='time_series'
)
print('\n指标表现评估结果:')
print(results_df.sort_values('sharpe_ratio', ascending=False))

## 5. CSCV过拟合检验

In [ ]:
# CSCV过拟合检验
cscv = CSCVTest(n_splits=50)

# 对EMA20进行CSCV检验
cscv_result = cscv.run_cscv_analysis(
    ema_20_signal_aligned,
    prices_df.mean(axis=1)
)

print('EMA(20) CSCV过拟合检验结果:')
print(f'  样本内夏普比率均值: {cscv_result["is_sharpe_mean"]:.4f}')
print(f'  样本外夏普比率均值: {cscv_result["oos_sharpe_mean"]:.4f}')
print(f'  过拟合概率: {cscv_result["overfitting_probability"]:.4f}')

In [ ]:
# 批量CSCV检验
robustness_analyzer = StrategyRobustnessAnalyzer()

robustness_results = robustness_analyzer.analyze_strategy_robustness(
    prices_df,
    sample_signals_aligned,
    strategy_type='time_series'
)

print('\n批量CSCV检验结果:')
print(robustness_results.sort_values('sharpe_ratio', ascending=False))

## 6. 可视化分析

In [ ]:
# 绘制价格走势图
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 主要指数价格走势
ax1 = axes[0, 0]
for col in ['沪深300', '上证50', '中证500']:
    if col in prices_df.columns:
        normalized = prices_df[col] / prices_df[col].iloc[0] * 100
        ax1.plot(normalized.index, normalized, label=col)
ax1.set_title('主要股票指数走势 (标准化)')
ax1.legend()
ax1.set_xlabel('日期')
ax1.set_ylabel('标准化价格')

# 2. 指标分布
ax2 = axes[0, 1]
sample_results = results_df['sharpe_ratio'].head(10)
ax2.barh(sample_results.index, sample_results.values)
ax2.set_title('不同指标的夏普比率')
ax2.set_xlabel('夏普比率')

# 3. 过拟合概率分布
ax3 = axes[1, 0]
ax3.scatter(robustness_results['sharpe_ratio'], 
           robustness_results['overfit_prob'],
           alpha=0.6)
ax3.axhline(y=0.5, color='r', linestyle='--', label='阈值 0.5')
ax3.set_title('夏普比率 vs 过拟合概率')
ax3.set_xlabel('夏普比率')
ax3.set_ylabel('过拟合概率')
ax3.legend()

# 4. 虚拟序列示例
ax4 = axes[1, 1]
for i in range(min(5, len(virtual_sequences))):
    seq = virtual_sequences[i]
    ax4.plot(range(len(seq['prices'])), seq['prices'].values, alpha=0.5)
ax4.set_title('蒙特卡洛虚拟序列示例')
ax4.set_xlabel('交易日')
ax4.set_ylabel('价格')

plt.tight_layout()
plt.savefig('../output/analysis_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('图表已保存至 output/analysis_results.png')

## 7. 策略总结

In [ ]:
# 输出研报核心结论对照
print('='*60)
print('研报核心结论复现总结')
print('='*60)

print('\n【结论1】趋势追踪指标具有"追涨杀跌"特性')
print('  - 本项目实现了41个趋势追踪指标')
print('  - 指标类型包括MA、EMA、MACD、ROC、DPO等')

print('\n【结论2】底层资产风险收益特性对策略表现有决定性影响')
print('  - 高收益率、低波动资产更适合趋势追踪策略')
print('  - 资产自相关性影响最大回撤')

print('\n【结论3】大类资产配置适合时序动量，行业配置适合截面动量')
print('  - 大类资产间相关性低，适合独立判断买入信号')
print('  - 行业指数相关性高，适合相对比较排名')

print('\n【结论4】CSCV过拟合概率低于50%的指标更可靠')
if len(robustness_results) > 0:
    reliable = robustness_results[robustness_results['overfit_prob'] < 0.5]
    print(f'  - 本次测试中 {len(reliable)}/{len(robustness_results)} 个指标过拟合概率低于50%')

## 注意事项

1. **数据限制**: 由于tushare接口限制，部分历史数据可能无法获取完整
2. **计算耗时**: 完整评估41个指标的所有参数组合需要较长时间
3. **研报差异**: 本项目为教学目的实现，与原研报方法论一致但参数细节可能有差异
4. **数据补充**: 如需完整复现，建议补充以下数据:
   - 中债国债指数历史数据
   - 南华商品指数数据
   - 完整的行业分类数据

In [ ]:
print('\n复现完成!')